# AQI Model Training Notebook
**Models:** Ridge · Lasso · Random Forest · XGBoost  
**Targets:** Current AQI · PM2.5 Day+1 · PM2.5 Day+2 · AQI Category Day+1 · Trend Direction  
> Feature engineering is handled by `feature_engineering.py` in the same folder.

## 0 · Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Sklearn models
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier

# Sklearn utilities
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, root_mean_squared_error, r2_score,
    f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import joblib

# Our feature engineering module
from feature_engineering import load_and_engineer, get_X_y, ALL_TARGETS

OUTPUT_DIR = Path('aqi_model_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Setup complete ✓')

## 1 · Load & Engineer Features

In [ ]:
DATA_PATH = r'data\historical_backup\historical_2026-02-09_2026-05-10.csv'

df = load_and_engineer(DATA_PATH)
df.head(3)

## 2 · Quick EDA on Engineered Features

In [ ]:
new_features = ['pressure_x_temp', 'wind_dispersion', 'dew_depression',
                'inversion_risk', 'pollution_persistence', 'aqi_trend_slope']
new_features = [f for f in new_features if f in df.columns]

target_col = 'target_aqi_current'
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Engineered Features vs target_aqi_current', fontsize=14, y=1.01)

for ax, feat in zip(axes.flatten(), new_features):
    mask = np.isfinite(df[feat]) & np.isfinite(df[target_col])
    x, y = df.loc[mask, feat].values, df.loc[mask, target_col].values
    r = np.corrcoef(x, y)[0, 1]
    ax.scatter(x, y, alpha=0.25, s=7, color='#2ecc71')
    try:
        m, b = np.polyfit(x, y, 1)
        ax.plot(np.linspace(x.min(), x.max(), 100),
                m * np.linspace(x.min(), x.max(), 100) + b, 'r-', lw=1.5)
    except Exception:
        pass
    ax.set_title(f'{feat}  (r={r:.2f})')
    ax.set_xlabel(feat); ax.set_ylabel('AQI')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'engineered_features_eda.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA plot saved.')

## 3 · Regression — target_aqi_current

In [ ]:
TARGET = 'target_aqi_current'
X_reg, y_reg, feat_names, _ = get_X_y(df, TARGET)
print(f'Feature matrix: {X_reg.shape}')

In [ ]:
# ── Define models
# Ridge & Lasso need scaling; RF & XGBoost don't
reg_models = {
    'Ridge':  Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=10.0))]),
    'Lasso':  Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=0.5, max_iter=5000))]),
    'RandomForest': RandomForestRegressor(
        n_estimators=300, max_depth=12, min_samples_leaf=3, n_jobs=-1, random_state=42),
    'XGBoost': XGBRegressor(
        n_estimators=500, learning_rate=0.04, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        tree_method='hist', verbosity=0, random_state=42),
}

tscv = TimeSeriesSplit(n_splits=5)
reg_results = []

for name, model in reg_models.items():
    maes, rmses, r2s = [], [], []
    for tr_idx, val_idx in tscv.split(X_reg):
        X_tr, X_val = X_reg.iloc[tr_idx], X_reg.iloc[val_idx]
        y_tr, y_val = y_reg[tr_idx], y_reg[val_idx]
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        maes.append(mean_absolute_error(y_val, preds))
        rmses.append(root_mean_squared_error(y_val, preds))
        r2s.append(r2_score(y_val, preds))
    reg_results.append({
        'Model': name,
        'MAE':  round(np.mean(maes), 3),
        'RMSE': round(np.mean(rmses), 3),
        'R²':   round(np.mean(r2s), 3),
    })
    print(f'  {name:15s} MAE={np.mean(maes):.2f}  RMSE={np.mean(rmses):.2f}  R²={np.mean(r2s):.3f}')

reg_df = pd.DataFrame(reg_results).sort_values('MAE')
reg_df

In [ ]:
# ── Plot model comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle(f'Regression Comparison — {TARGET}', fontsize=13)
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for ax, metric in zip(axes, ['MAE', 'RMSE', 'R²']):
    bars = ax.bar(reg_df['Model'], reg_df[metric], color=colors)
    ax.set_title(metric); ax.set_ylabel(metric)
    for bar, val in zip(bars, reg_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'comparison_reg_{TARGET}.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature importance for tree models
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, name in zip(axes, ['RandomForest', 'XGBoost']):
    model = reg_models[name]
    model.fit(X_reg, y_reg)   # refit on full data
    imp = pd.Series(model.feature_importances_, index=feat_names).nlargest(20)
    imp.sort_values().plot.barh(ax=ax, color='#3498db')
    ax.axvline(imp.mean(), color='red', ls='--', alpha=0.6, label='mean')
    ax.set_title(f'{name} — Top 20 Features')
    ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'feat_imp_reg_{TARGET}.png', dpi=120, bbox_inches='tight')
plt.show()

# Lasso coefficients (non-zero = selected features)
lasso_model = reg_models['Lasso'].fit(X_reg, y_reg)
lasso_coefs = pd.Series(
    lasso_model.named_steps['model'].coef_, index=feat_names
).sort_values(key=abs, ascending=False)
selected = lasso_coefs[lasso_coefs != 0]
print(f'\nLasso selected {len(selected)}/{len(feat_names)} features:')
print(selected.round(4).to_string())

## 4 · Regression — PM2.5 Day+1 & Day+2 Forecasts

In [ ]:
forecast_results = {}

for TARGET_F in ['target_pm25_day1_avg', 'target_pm25_day2_avg']:
    X_f, y_f, feat_f, _ = get_X_y(df, TARGET_F)
    print(f'\n── {TARGET_F} ({X_f.shape[1]} features) ──')

    forecast_models = {
        'Ridge':  Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=10.0))]),
        'Lasso':  Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=0.5, max_iter=5000))]),
        'RandomForest': RandomForestRegressor(
            n_estimators=300, max_depth=12, min_samples_leaf=3, n_jobs=-1, random_state=42),
        'XGBoost': XGBRegressor(
            n_estimators=500, learning_rate=0.04, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            tree_method='hist', verbosity=0, random_state=42),
    }

    rows = []
    for name, model in forecast_models.items():
        maes, r2s = [], []
        for tr_idx, val_idx in TimeSeriesSplit(n_splits=5).split(X_f):
            model.fit(X_f.iloc[tr_idx], y_f[tr_idx])
            preds = model.predict(X_f.iloc[val_idx])
            maes.append(mean_absolute_error(y_f[val_idx], preds))
            r2s.append(r2_score(y_f[val_idx], preds))
        rows.append({'Model': name, 'MAE': round(np.mean(maes),3), 'R²': round(np.mean(r2s),3)})
        print(f'  {name:15s} MAE={np.mean(maes):.2f}  R²={np.mean(r2s):.3f}')

    forecast_results[TARGET_F] = pd.DataFrame(rows).sort_values('MAE')

forecast_results['target_pm25_day1_avg']

## 5 · Classification — AQI Category Day+1

In [ ]:
CLF_TARGET = 'target_aqi_cat_day1'
X_clf, y_clf, feat_clf, le_clf = get_X_y(df, CLF_TARGET)
n_classes = len(le_clf.classes_)
avg_type  = 'binary' if n_classes == 2 else 'weighted'
print(f'Classes: {dict(enumerate(le_clf.classes_))}')

In [ ]:
obj = 'binary:logistic' if n_classes == 2 else 'multi:softmax'

clf_models = {
    'Ridge (LogReg)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', __import__('sklearn.linear_model', fromlist=['LogisticRegression'])
                  .LogisticRegression(C=0.1, max_iter=1000, solver='lbfgs',
                                      multi_class='auto', random_state=42))
    ]),
    'Lasso (LogReg)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', __import__('sklearn.linear_model', fromlist=['LogisticRegression'])
                  .LogisticRegression(C=0.1, penalty='l1', solver='saga',
                                      max_iter=1000, random_state=42))
    ]),
    'RandomForest': RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=3, n_jobs=-1, random_state=42),
    'XGBoost': XGBClassifier(
        n_estimators=500, learning_rate=0.04, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, objective=obj,
        tree_method='hist', verbosity=0, eval_metric='logloss', random_state=42),
}

clf_results = []
for name, model in clf_models.items():
    f1s, accs = [], []
    for tr_idx, val_idx in TimeSeriesSplit(n_splits=5).split(X_clf):
        model.fit(X_clf.iloc[tr_idx], y_clf[tr_idx])
        preds = model.predict(X_clf.iloc[val_idx])
        f1s.append(f1_score(y_clf[val_idx], preds, average=avg_type, zero_division=0))
        accs.append((preds == y_clf[val_idx]).mean())
    clf_results.append({'Model': name,
                        'F1': round(np.mean(f1s), 3),
                        'Accuracy': round(np.mean(accs), 3)})
    print(f'  {name:18s} F1={np.mean(f1s):.3f}  Acc={np.mean(accs):.3f}')

clf_df = pd.DataFrame(clf_results).sort_values('F1', ascending=False)
clf_df

In [ ]:
# ── Confusion matrix for best model
best_clf_name = clf_df.iloc[0]['Model']
best_clf = clf_models[best_clf_name]

splits = list(TimeSeriesSplit(n_splits=5).split(X_clf))
tr_idx, val_idx = splits[-1]
best_clf.fit(X_clf.iloc[tr_idx], y_clf[tr_idx])
preds = best_clf.predict(X_clf.iloc[val_idx])

print(f'\nBest model: {best_clf_name}')
print(classification_report(y_clf[val_idx], preds,
                             target_names=le_clf.classes_, zero_division=0))

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_clf[val_idx], preds),
    display_labels=le_clf.classes_
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — {CLF_TARGET}\n({best_clf_name})')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'confusion_{CLF_TARGET}.png', dpi=120, bbox_inches='tight')
plt.show()

## 6 · Classification — Trend Direction

In [ ]:
TREND_TARGET = 'target_trend_direction'
X_tr_d, y_tr_d, feat_tr, le_tr = get_X_y(df, TREND_TARGET)
n_tr = len(le_tr.classes_)
avg_tr = 'binary' if n_tr == 2 else 'weighted'
print(f'Classes: {dict(enumerate(le_tr.classes_))}')

obj_tr = 'binary:logistic' if n_tr == 2 else 'multi:softmax'
trend_models = {
    'Ridge (LogReg)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', __import__('sklearn.linear_model', fromlist=['LogisticRegression'])
                  .LogisticRegression(C=0.1, max_iter=1000, random_state=42))
    ]),
    'Lasso (LogReg)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', __import__('sklearn.linear_model', fromlist=['LogisticRegression'])
                  .LogisticRegression(C=0.1, penalty='l1', solver='saga',
                                      max_iter=1000, random_state=42))
    ]),
    'RandomForest': RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=3, n_jobs=-1, random_state=42),
    'XGBoost': XGBClassifier(
        n_estimators=500, learning_rate=0.04, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, objective=obj_tr,
        tree_method='hist', verbosity=0, eval_metric='logloss', random_state=42),
}

trend_results = []
for name, model in trend_models.items():
    f1s = []
    for tr_idx, val_idx in TimeSeriesSplit(n_splits=5).split(X_tr_d):
        model.fit(X_tr_d.iloc[tr_idx], y_tr_d[tr_idx])
        preds = model.predict(X_tr_d.iloc[val_idx])
        f1s.append(f1_score(y_tr_d[val_idx], preds, average=avg_tr, zero_division=0))
    trend_results.append({'Model': name, 'F1': round(np.mean(f1s), 3)})
    print(f'  {name:18s} F1={np.mean(f1s):.3f}')

trend_df = pd.DataFrame(trend_results).sort_values('F1', ascending=False)
print(f'\nDecoded class labels:')
for i, label in enumerate(le_tr.classes_):
    print(f'  {i} → {label}')
trend_df

## 7 · Save Best Models

In [ ]:
# Retrain best models on full data and save
saved = []

# Best regression
best_reg_name = reg_df.iloc[0]['Model']
best_reg = reg_models[best_reg_name]
best_reg.fit(X_reg, y_reg)
path = OUTPUT_DIR / f'best_reg_target_aqi_current_{best_reg_name}.pkl'
joblib.dump({'model': best_reg, 'features': feat_names}, path)
saved.append(str(path))

# Best classification
best_clf.fit(X_clf, y_clf)
path = OUTPUT_DIR / f'best_clf_{CLF_TARGET}_{best_clf_name}.pkl'
joblib.dump({'model': best_clf, 'label_encoder': le_clf, 'features': feat_clf}, path)
saved.append(str(path))

print('Saved models:')
for s in saved:
    print(f'  ✔ {s}')

## 8 · Final Summary Table

In [ ]:
print('='*60)
print('REGRESSION RESULTS — target_aqi_current')
print('='*60)
print(reg_df.to_string(index=False))

print('\n' + '='*60)
print('REGRESSION RESULTS — PM2.5 Forecasts')
print('='*60)
for t, res in forecast_results.items():
    print(f'\n{t}:')
    print(res.to_string(index=False))

print('\n' + '='*60)
print('CLASSIFICATION — AQI Category Day+1')
print('='*60)
print(clf_df.to_string(index=False))

print('\n' + '='*60)
print('CLASSIFICATION — Trend Direction')
print('='*60)
print(trend_df.to_string(index=False))